In [ ]:
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer


print("=" * 60)
print("VQE for N2 Molecule on simulator")
print("=" * 60)

# ---- Step 1: Generate N2 Hamiltonian ----
print("\n[1/5] Generating N2 Hamiltonian...")
driver = PySCFDriver(atom='N 0 0 0; N 0 0 1.098', basis='cc-pVDZ')

active_space = ActiveSpaceTransformer(
    num_electrons=(5, 5),  # 10 - 4 frozen = 6 aktivních elektronů
    num_spatial_orbitals=8

)

freeze_core = FreezeCoreTransformer()

problem = driver.run()
problem = freeze_core.transform(problem)

problem_reduced = active_space.transform(problem)
mapper = JordanWignerMapper()
hamiltonian = mapper.map(problem_reduced.hamiltonian.second_q_op())

print(f"  ✓ Number of qubits: {hamiltonian.num_qubits}")
print(f"  ✓ Number of Pauli terms: {len(hamiltonian)}")

In [ ]:
from scipy.sparse.linalg import eigsh

# Sparse reprezentace
hamiltonian_sparse = hamiltonian.to_matrix(sparse=True)

# Najdi nejnižší vlastní hodnotu
eigenvalues, eigenvectors = eigsh(hamiltonian_sparse, k=1, which='SA')
ground_state_energy = eigenvalues[0]

print(f"Ground state energy: {ground_state_energy} Ha")
print(f"Ground state vector shape: {eigenvectors.shape}")

In [ ]:
# Nuclear repulsion energy for N2 at 1.098 Å
Z_N = 7  # Atomic number from periodic table
R = 1.098  # Å
R_bohr = R * 1.88973  # Conversion: 1 Å = 1.88973 Bohr
E_nuc = (Z_N * Z_N) / R_bohr  # Nuclear repulsion energy in Ha

print(f"  ✓ Nuclear repulsion energy: {E_nuc:.6f} Ha")

In [ ]:
# Create Hartree-Fock initial state
from qiskit_nature.second_q.circuit.library import HartreeFock

hf_state = HartreeFock(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper
)


In [ ]:
# Pre-defined ansatz circuit and operator class for Hamiltonian
from qiskit.circuit.library import efficient_su2
 
# Note that it is more common to place initial 'h' gates outside the ansatz. Here we specifically wanted this layer structure.
ansatz = efficient_su2(
    hamiltonian.num_qubits, su2_gates=["rz", "y"], entanglement="linear", reps=1
)
num_params = ansatz.num_parameters
print("This circuit has ", num_params, "parameters")
 
ansatz.decompose().draw("mpl", style="iqp")

In [ ]:
from scipy.optimize import minimize
from qiskit_aer.primitives import EstimatorV2 as Estimator
import numpy as np

iterations = [0]
def cost_function(params, ansatyz, estimator2=Estimator):
    estimator = estimator2
    iterations[0] += 1
    print(f"Iteration {iterations[0]}: Evaluating cost function...")
    hf_circuit = hf_state.compose(ansatz)
    bound_circuit = hf_circuit.assign_parameters(params)
    job = estimator.run([(bound_circuit, hamiltonian)])
    result = job.result()
    electronic = result[0].data.evs
    print(f"  Energy: {electronic + problem.nuclear_repulsion_energy:.6f} Ha")


    return electronic + problem.nuclear_repulsion_energy

# Optimize!
initial_params = np.zeros(ansatz.num_parameters)

In [ ]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Session,
    EstimatorV2 as Estimator,
)
 
service = QiskitRuntimeService()

backend = service.least_busy(operational=True, simulator=False)

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3
)

ansatz = hf_state.compose(ansatz)


pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_ansatz = pm.run(ansatz)
isa_hamiltonian = hamiltonian.apply_layout(isa_ansatz.layout)


def cost_function(params, ansatz, estimator):
    iterations[0] += 1
    print(f"Iteration {iterations[0]}")

    bound = ansatz.assign_parameters(params)

    job = estimator.run([(bound, isa_hamiltonian)])
    result = job.result()
    electronic = result[0].data.evs

    return electronic + problem.nuclear_repulsion_energy

In [ ]:
print("Depth:", isa_ansatz.depth())
print("2q gates:", isa_ansatz.num_nonlocal_gates())
print("Pauli terms:", len(isa_hamiltonian.paulis))

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from qiskit_ibm_runtime import EstimatorV2 as Estimator
 
 
def minimize_thread(estimator, method):
    return minimize(
        cost_function,
        x0=np.zeros(isa_ansatz.num_parameters),
        args=(isa_ansatz, estimator),
        method=method,
        options={"maxiter": 250},
    )
 
 
with Session(backend=backend), ThreadPoolExecutor() as executor:
    estimator1 = Estimator()
    estimator1.options.default_precision = 0.05
 
    # Use different tags to differentiate the jobs.
    estimator1.options.environment.job_tags = ["cobyla"]
 
    # Submit the  workload.
    cobyla_future = executor.submit(minimize_thread, estimator1, "cobyla")

    # Get workload results.
    cobyla_result = cobyla_future.result()
